# Assignment 8: Build the Intent Dataset for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 02nd November 2026, 23:59**

Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
do the checking, the split, the baseline and the entity count: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Week 7 example) with only 6 messages
per label, so the quality report shows what is still missing. It may not be submitted.

**Steps:** Save a copy of this file in Drive. Replace Steps 1, 2, 3, 6 and 7 with your own, and run
everything from the top (Runtime, then Run all). Repeat Steps 3 and 4 until the report is clean.

---

---

## Setup

**PROVIDED.** Run once. The same spaCy model as the session notebook.

In [21]:
# AITCD001, Week 8 take home: the intent dataset for my own chatbot
import csv
import random
import re
from collections import Counter

import pandas as pd
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:                  # not installed yet: install it once
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("ready: spaCy", spacy.__version__, "| model", nlp.meta["version"])

ready: spaCy 3.8.18 | model 3.8.0


---

## Step 1: The chatbot and its labels

**YOUR DOMAIN.** The chatbot of your Portfolio Project 1. All of its intents, plus `out_of_scope`. Do not add `greeting` or `name` unless your users really ask for something with them.

In [22]:
# STEP 1. YOUR DOMAIN: the chatbot of your Portfolio Project 1, and its labels.
BOT_NAME = "ULK Exam & Registry Desk"
INTENTS = ["timetable", "fees", "transcript", "book", "cancel", "out_of_scope"]
TARGET = 30          # messages per label, at least

print(BOT_NAME, "|", len(INTENTS), "labels:", INTENTS)

ULK Exam & Registry Desk | 6 labels: ['timetable', 'fees', 'transcript', 'book', 'cancel', 'out_of_scope']


---

## Step 2: The labelling guideline

**YOUR DOMAIN.** Write the guideline BEFORE labelling. Every time two labels are possible for a message, add or sharpen a rule.

In [23]:
# STEP 2. YOUR DOMAIN: your labelling guideline.
# Each rule settles a case where two people could disagree. Copy it into the README.
GUIDELINE = [
    "One message, one label. If a message contains two separate requests, split it first.",
    "Label the user's goal, not a single keyword: 'how much is a transcript' asks for price, so it is fees.",
    "Any request to schedule, reserve, or get a slot at the registry is book, even if it names a service like transcript or clearance.",
    "Asking when, where, or how long it takes to collect a transcript or check grades/marks is transcript.",
    "Asking for exam dates, times, schedules, or sessions is timetable.",
    "Complaints, lost student IDs/items, lecturer contacts, or general university admissions/news are out_of_scope.",
    "Keep the student's exact words: preserve typos (like 'cancle'), SMS shorthand, and Kinyarwanda or mixed phrasing as typed.",
]

for i, rule in enumerate(GUIDELINE, start=1):
    print(i, rule)

1 One message, one label. If a message contains two separate requests, split it first.
2 Label the user's goal, not a single keyword: 'how much is a transcript' asks for price, so it is fees.
3 Any request to schedule, reserve, or get a slot at the registry is book, even if it names a service like transcript or clearance.
4 Asking when, where, or how long it takes to collect a transcript or check grades/marks is transcript.
5 Asking for exam dates, times, schedules, or sessions is timetable.
6 Complaints, lost student IDs/items, lecturer contacts, or general university admissions/news are out_of_scope.
7 Keep the student's exact words: preserve typos (like 'cancle'), SMS shorthand, and Kinyarwanda or mixed phrasing as typed.


---

## Step 3: The dataset

**YOUR DOMAIN.** At least 30 messages per label. Sources, in order of value: your chatbot's UNMATCHED log and Step 10 local phrasing from Portfolio Project 1; at least 20 messages from real people you ask (classmates, family, friends, for example on WhatsApp: *how would you ask a bus desk about luggage?*); then your own. Keep every message exactly as it was typed.

In [24]:
# STEP 3. YOUR DOMAIN: the dataset. Each row is (message, label, source).
# source is one of: "rules", "unmatched", "local", "people", "written"
DATA = [
    # --- TIMETABLE (30 rows: 3 rules, 6 local, 4 people, 17 written) ---
    ("when are exams", "timetable", "rules"),
    ("check exam timetable", "timetable", "rules"),
    ("what time is the exam", "timetable", "rules"),
    ("ibizamini ni ryari", "timetable", "local"),
    ("ibizamini bitangira ryari", "timetable", "local"),
    ("igikorwa cy ibizamini ni ryari", "timetable", "local"),
    ("ni ryari ibizamini bizaba", "timetable", "local"),
    ("saa ngahe ikizamini gitangira", "timetable", "local"),
    ("ndabaza igihe ibizamini bizabera", "timetable", "local"),
    ("Hi, exams zitangira ryari?", "timetable", "people"),
    ("Nzakora morning or afternoon session?", "timetable", "people"),
    ("hello pls when do final exams start?", "timetable", "people"),
    ("i want to know my session", "timetable", "people"),
    ("exam schedule for this semester", "timetable", "written"),
    ("morning or afternoon exam session", "timetable", "written"),
    ("when do main exams start", "timetable", "written"),
    ("show me the exam timetable", "timetable", "written"),
    ("what time does morning exam begin", "timetable", "written"),
    ("exam timetable pls", "timetable", "written"),
    ("wen r exams starting", "timetable", "written"),
    ("is there an afternoon exam session", "timetable", "written"),
    ("dates for final exams", "timetable", "written"),
    ("what days do exams run", "timetable", "written"),
    ("exam hours at ulk", "timetable", "written"),
    ("schedule of exams", "timetable", "written"),
    ("when is afternoon paper", "timetable", "written"),
    ("can i see the exam dates", "timetable", "written"),
    ("time for morning exams", "timetable", "written"),
    ("do exams run on friday", "timetable", "written"),
    ("wht time is afternoon exam", "timetable", "written"),

    # --- FEES (30 rows: 3 rules, 6 local, 4 people, 17 written) ---
    ("how much is clearance", "fees", "rules"),
    ("clearance fee", "fees", "rules"),
    ("price for transcript", "fees", "rules"),
    ("clearance ni angahe", "fees", "local"),
    ("hw much 4 transcript", "fees", "local"),
    ("amafaranga ya clearance ni angahe", "fees", "local"),
    ("ni angahe kwishyura transcript", "fees", "local"),
    ("bishyura angahe kuri clearance", "fees", "local"),
    ("amafaranga yo gusubiramo ikizamini ni angahe", "fees", "local"),
    ("Muraho, clearance ni angahe?", "fees", "people"),
    ("And transcript?", "fees", "people"),
    ("how much for resit exam?", "fees", "people"),
    ("n’ID nshya if i lost mine?", "fees", "people"),
    ("how much do i pay for resit", "fees", "written"),
    ("cost of student id replacement", "fees", "written"),
    ("how much is re-sit exam", "fees", "written"),
    ("fee for lost id card", "fees", "written"),
    ("what is the clearance charge", "fees", "written"),
    ("transcript costs how much", "fees", "written"),
    ("how do i pay clearance fees", "fees", "written"),
    ("resit fee amount", "fees", "written"),
    ("price of new student id", "fees", "written"),
    ("how much for academic clearance", "fees", "written"),
    ("fees for registry services", "fees", "written"),
    ("hw mch is resit", "fees", "written"),
    ("cost of getting a transcript", "fees", "written"),
    ("id replacement fee", "fees", "written"),
    ("how much does clearance cost", "fees", "written"),
    ("what are the resit charges", "fees", "written"),
    ("how much to pay on irembo for clearance", "fees", "written"),

    # --- TRANSCRIPT (30 rows: 3 rules, 6 local, 4 people, 17 written) ---
    ("where do i get my transcript", "transcript", "rules"),
    ("check my exam results", "transcript", "rules"),
    ("get my grades", "transcript", "rules"),
    ("amanota yanjye", "transcript", "local"),
    ("ndabaza amanota yanjye", "transcript", "local"),
    ("amanota arasohoka ryari", "transcript", "local"),
    ("ndasaba transcript yanjye", "transcript", "local"),
    ("ndifuza kureba amanota", "transcript", "local"),
    ("kureba amanota y ibizamini", "transcript", "local"),
    ("Hi, nashakaga kureba marks zanjye.", "transcript", "people"),
    ("Transcript nyifata he and when?", "transcript", "people"),
    ("pls can i check my grades?", "transcript", "people"),
    ("when is my transcript ready 4 pickup", "transcript", "people"),
    ("how long does transcript take", "transcript", "written"),
    ("where is the academic registry", "transcript", "written"),
    ("collect my provisional transcript", "transcript", "written"),
    ("when will my transcript be ready", "transcript", "written"),
    ("which room issues transcripts", "transcript", "written"),
    ("how many days for transcript", "transcript", "written"),
    ("where can i pick up my marks", "transcript", "written"),
    ("transcript collection office", "transcript", "written"),
    ("how to get my transcript after clearance", "transcript", "written"),
    ("is my transcript ready", "transcript", "written"),
    ("where to collect results", "transcript", "written"),
    ("provisional transcript collection", "transcript", "written"),
    ("what block is registry for transcripts", "transcript", "written"),
    ("whr do i collect transcript", "transcript", "written"),
    ("how do i check my marks", "transcript", "written"),
    ("registry room number for transcript", "transcript", "written"),
    ("can i pick up my transcript today", "transcript", "written"),

    # --- BOOK (30 rows: 3 rules, 6 local, 4 people, 17 written) ---
    ("book an appointment", "book", "rules"),
    ("book registry slot", "book", "rules"),
    ("reserve an appointment", "book", "rules"),
    ("book registry 2moro", "book", "local"),
    ("ndashaka gufata gahunda kuri registry", "book", "local"),
    ("fata gahunda ya clearance", "book", "local"),
    ("ndasaba gahunda yo kubonana na registry", "book", "local"),
    ("gufata gahunda ya transcript", "book", "local"),
    ("ndashaka slot kuri registry ejo", "book", "local"),
    ("Hello, nshaka gufata appointment ku Academic Registry tomorrow morning pls", "book", "people"),
    ("hi book me a slot today afternoon at registry", "book", "people"),
    ("can i reserve a morning appointment tomorrow", "book", "people"),
    ("book me for clearance at registry today", "book", "people"),
    ("i want to book a transcript appointment", "book", "written"),
    ("schedule a slot for clearance", "book", "written"),
    ("can i book for tomorrow morning", "book", "written"),
    ("book an id appointment today", "book", "written"),
    ("reserve a morning slot", "book", "written"),
    ("i need an appointment at registry", "book", "written"),
    ("book resit appointment", "book", "written"),
    ("can i get a slot for afternoon", "book", "written"),
    ("schedule registry visit tomorrow", "book", "written"),
    ("i want to reserve a registry slot", "book", "written"),
    ("make an appointment for transcript", "book", "written"),
    ("book slot 4 tomorrow", "book", "written"),
    ("book afternoon session", "book", "written"),
    ("schedule an appointment for id", "book", "written"),
    ("book registry for resit", "book", "written"),
    ("book morning appointment today", "book", "written"),
    ("bk appointment 2moro", "book", "written"),

    # --- CANCEL (30 rows: 3 rules, 1 unmatched, 5 local, 4 people, 17 written) ---
    ("cancel my appointment", "cancel", "rules"),
    ("call it off", "cancel", "rules"),
    ("stop the booking", "cancel", "rules"),
    ("cancle", "cancel", "unmatched"),
    ("ndashaka guhagarika gahunda", "cancel", "local"),
    ("hagarika gahunda nari nafashwe", "cancel", "local"),
    ("kuraho gahunda yanjye", "cancel", "local"),
    ("ndahagaritse gahunda", "cancel", "local"),
    ("rekera aho gufata gahunda", "cancel", "local"),
    ("Muraho, nashatse guhagarika appointment nafashe.", "cancel", "people"),
    ("Please cancel it", "cancel", "people"),
    ("hey i need to cancel my registry appointment, sinshobora kuza", "cancel", "people"),
    ("sinzaboneka ejo cancel my slot", "cancel", "people"),
    ("cancel booking", "cancel", "written"),
    ("i want to cancel my slot", "cancel", "written"),
    ("abort appointment", "cancel", "written"),
    ("cancel registry appointment", "cancel", "written"),
    ("dont book it anymore", "cancel", "written"),
    ("please cancel my reservation", "cancel", "written"),
    ("stop booking", "cancel", "written"),
    ("i changed my mind cancel", "cancel", "written"),
    ("cancel the slot", "cancel", "written"),
    ("call off my registry appointment", "cancel", "written"),
    ("cncl booking", "cancel", "written"),
    ("no longer need the appointment", "cancel", "written"),
    ("cancel tomorrow appointment", "cancel", "written"),
    ("drop my booking", "cancel", "written"),
    ("undo my booking", "cancel", "written"),
    ("cancel my clearance slot", "cancel", "written"),
    ("remove my appointment", "cancel", "written"),

    # --- OUT_OF_SCOPE (30 rows: 2 unmatched, 3 local, 4 people, 21 written) ---
    ("asdfgh", "out_of_scope", "unmatched"),
    ("qwerty", "out_of_scope", "unmatched"),
    ("amafaranga y ishuri ni angahe", "out_of_scope", "local"),
    ("bibiliyoteke ifungura ryari", "out_of_scope", "local"),
    ("ikibuga cy umupira kiri hehe", "out_of_scope", "local"),
    ("Hi, Wi-Fi ya campus ntikora, mwayikosora?", "out_of_scope", "people"),
    ("what time does the library close?", "out_of_scope", "people"),
    ("and i lost my phone, anyone found it?", "out_of_scope", "people"),
    ("tuition fee ya semester ni angahe?", "out_of_scope", "people"),
    ("i lost my student card", "out_of_scope", "written"),
    ("what is the wifi password", "out_of_scope", "written"),
    ("who is the dean of faculty", "out_of_scope", "written"),
    ("where is the library", "out_of_scope", "written"),
    ("how do i apply for admission", "out_of_scope", "written"),
    ("my lecturer is not in class", "out_of_scope", "written"),
    ("is the cafeteria open", "out_of_scope", "written"),
    ("where can i park my car", "out_of_scope", "written"),
    ("i need a medical excuse letter", "out_of_scope", "written"),
    ("phone number of the rector", "out_of_scope", "written"),
    ("when does next semester start", "out_of_scope", "written"),
    ("tuition fees for year two", "out_of_scope", "written"),
    ("my portal password is blocked", "out_of_scope", "written"),
    ("where is block c", "out_of_scope", "written"),
    ("are there hostels at ulk", "out_of_scope", "written"),
    ("who won the football match", "out_of_scope", "written"),
    ("tell me a joke", "out_of_scope", "written"),
    ("i forgot my moodle login", "out_of_scope", "written"),
    ("is there graduation this year", "out_of_scope", "written"),
    ("how to get a scholarship", "out_of_scope", "written"),
    ("where is the finance office", "out_of_scope", "written"),
]

print(len(DATA), "rows")

180 rows


---

## Step 4: The quality report

**PROVIDED.** Read every line. Fix the data in Step 3 until every label is ok, there are no duplicates and no conflicts, and the balance is 1.5 or less. Then run Step 3 and Step 4 again.

In [25]:
# STEP 4. PROVIDED: the quality report. Read it, fix the data in Step 3, run again.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


per_label = Counter()
per_source = Counter()
unknown = []
for text, label, source in DATA:
    per_label[label] += 1
    per_source[source] += 1
    if label not in INTENTS:
        unknown.append((text, label))

print("Per label:")
for label in INTENTS:
    n = per_label[label]
    note = "ok" if n >= TARGET else f"needs more, target {TARGET}"
    print(f"   {label:14}{n:4}   {note}")
print("Per source:", dict(per_source))
print("people:", per_source["people"], "of at least 20")

seen = {}
duplicates = []
conflicts = []
for text, label, source in DATA:
    key = clean(text)
    if key in seen:
        duplicates.append(text)
        if seen[key] != label:
            conflicts.append(text)
    seen[key] = label

print("Labels not in INTENTS:", unknown)
print("Duplicates:", duplicates)
print("Conflicting labels:", conflicts)
counts = [per_label[label] for label in INTENTS]
print("Balance, largest / smallest:", round(max(counts) / max(1, min(counts)), 2))

Per label:
   timetable       30   ok
   fees            30   ok
   transcript      30   ok
   book            30   ok
   cancel          30   ok
   out_of_scope    30   ok
Per source: {'rules': 15, 'local': 32, 'people': 24, 'written': 106, 'unmatched': 3}
people: 24 of at least 20
Labels not in INTENTS: []
Duplicates: []
Conflicting labels: []
Balance, largest / smallest: 1.0


---

## Step 5: The split, and the CSV file

**PROVIDED.** 20% of each label becomes the test set. The file `intents.csv` appears in the Files panel on the left: download it, and commit it to GitHub.

In [26]:
# STEP 5. PROVIDED: the 80/20 split, intent by intent, with a fixed seed; then the CSV.
rng = random.Random(8)

by_label = {}
for row in DATA:
    if row[1] not in by_label:
        by_label[row[1]] = []
    by_label[row[1]].append(row)

TRAIN = []
TEST = []
for label in sorted(by_label):
    rows = by_label[label][:]
    rng.shuffle(rows)
    cut = len(rows) // 5
    TEST.extend(rows[:cut])
    TRAIN.extend(rows[cut:])

with open("intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, label, source in TRAIN:
        writer.writerow([text, label, source, "train"])
    for text, label, source in TEST:
        writer.writerow([text, label, source, "test"])

df = pd.read_csv("intents.csv")
print(len(TRAIN), "training,", len(TEST), "test, saved to intents.csv")
print(pd.crosstab(df["intent"], df["split"]))

144 training, 36 test, saved to intents.csv
split         test  train
intent                   
book             6     24
cancel           6     24
fees             6     24
out_of_scope     6     24
timetable        6     24
transcript       6     24


---

## Step 6: The baseline: your rules on your test set

**YOUR DOMAIN.** Paste the rule table of your own chatbot. The baseline is the number your classifier must beat in Week 9. Every line it prints is a message the rules get wrong.

In [27]:
# STEP 6. YOUR DOMAIN: paste the RULES list from Step 2 of your Portfolio Project 1.
RULES = [
    ("cancel",     r"\bcancel\w*\b|\bcall it off\b|\bstop\b"),
    ("book",       r"\bbook\w*\b|\bappointments?\b|\breserv\w*\b|\bregistry\b|\bslot\b"),
    ("timetable",  r"\btimetables?\b|\bexam\w*\b|\bschedules?\b|\bwhat ?time\b|\bibizamini\b"),
    ("fees",       r"\bfees?\b|\bclearance\b|\bprices?\b|\bcosts?\b|\bhow ?much\b|\bangahe\b"),
    ("transcript", r"\btranscripts?\b|\bresults?\b|\bgrades?\b|\bmarks?\b|\bamanota\b"),
]

print(len(RULES), "rules")

5 rules


In [28]:
# STEP 6, continued. PROVIDED: the baseline, your rules on your test set.
def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


right = 0
for text, label, source in TEST:
    guess = rule_label(text)
    if guess == label:
        right += 1
    else:
        print(f"   {text!r}: labelled {label}, rules say {guess}")
print(f"Baseline: {right} of {len(TEST)} test messages right")

   'fata gahunda ya clearance': labelled book, rules say fees
   'Muraho, nashatse guhagarika appointment nafashe.': labelled cancel, rules say book
   'hagarika gahunda nari nafashwe': labelled cancel, rules say out_of_scope
   'kuraho gahunda yanjye': labelled cancel, rules say out_of_scope
   'n’ID nshya if i lost mine?': labelled fees, rules say out_of_scope
   'tuition fees for year two': labelled out_of_scope, rules say fees
   'what block is registry for transcripts': labelled transcript, rules say book
   'kureba amanota y ibizamini': labelled transcript, rules say timetable
   'how to get my transcript after clearance': labelled transcript, rules say fees
Baseline: 27 of 36 test messages right


---

## Step 7: The entities of your domain

**YOUR DOMAIN.** All entity labels of your own, each with the closed list of names your users mention: places, products, services, days. The provided part counts how often they appear in your dataset.

In [29]:
# STEP 7. YOUR DOMAIN: the entities of your domain. At least 2 labels of your own.
ENTITY_LISTS = {
    "SERVICE": ["clearance", "transcript", "resit", "re-sit", "id"],
    "DAY":     ["today", "tomorrow", "friday"],
    "SESSION": ["morning", "afternoon"],
}

patterns = []
for label in ENTITY_LISTS:
    for name in ENTITY_LISTS[label]:
        patterns.append({"label": label, "pattern": [{"LOWER": name}]})

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(patterns)

# PROVIDED: how many of your messages contain each entity, and five examples
found = Counter()
examples = []
for text, label, source in DATA:
    ents = [(e.text, e.label_) for e in nlp(text.lower()).ents]
    for e in ents:
        found[e[1]] += 1
    if ents and len(examples) < 5:
        examples.append((text, ents))

print("Entities found in the dataset:", dict(found))
for text, ents in examples:
    print("  ", text, "->", ents)

Entities found in the dataset: {'PERSON': 24, 'ORG': 9, 'SESSION': 17, 'DAY': 12, 'SERVICE': 47, 'CARDINAL': 5, 'NORP': 2, 'GPE': 1, 'DATE': 3}
   ibizamini ni ryari -> [('ibizamini ni', 'PERSON'), ('ryari', 'ORG')]
   ibizamini bitangira ryari -> [('ibizamini bitangira', 'PERSON'), ('ryari', 'ORG')]
   igikorwa cy ibizamini ni ryari -> [('ryari', 'ORG')]
   ni ryari ibizamini bizaba -> [('ryari ibizamini bizaba', 'ORG')]
   saa ngahe ikizamini gitangira -> [('ikizamini gitangira', 'PERSON')]


---

## Step 8: Publish and submit

1. In the repository of your Portfolio Project 1, create a folder named `dataset`.
2. Upload this notebook (File, then Download, then .ipynb) and `intents.csv` into it.
3. Add `dataset/README.md` from `README_template_dataset.md`, every section filled in.
4. Open the folder in a private browsing window. Paste its URL in Google Classroom and click
   **Turn in**, by **Monday 02 November 2026, 23:59**.

Never put real names, phone numbers or ID numbers in the dataset: it is public.